# 📂 EATD + MODMA Processing (Account 3)

## MODMA Dataset Info
| Attribute | Value |
|-----------|-------|
| MDD Subjects | 23 |
| Healthy Controls | 29 |
| Total | 52 participants |
| Age Range | 18-52 years |
| Size | 2.5 GB |
| Contents | Demographics + Psychological assessments |

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/EATD-Corpus"
!mkdir -p "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/MODMA"
!mkdir -p "/content/drive/MyDrive/DAIC-WOZ_Datasets/MODMA_Raw"
print("✅ Drive mounted")

## Step 2: Install Dependencies

In [ ]:
!pip install openai-whisper torch torchaudio h5py pandas tqdm gdown --quiet
!apt-get install -y ffmpeg unzip > /dev/null 2>&1
print("✅ Dependencies installed")

## Step 3: Setup Common Functions

In [ ]:
import whisper
import torch
import torchaudio
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

print("Loading Whisper model...")
whisper_model = whisper.load_model("base")
print("✅ Whisper loaded")

def resample_audio(path, target_sr=16000):
    waveform, sr = torchaudio.load(str(path))
    if sr != target_sr:
        waveform = torchaudio.transforms.Resample(sr, target_sr)(waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    return waveform.squeeze().numpy(), target_sr

def extract_features(waveform, sr=16000):
    chunk_size = sr * 10
    n_chunks = max(1, len(waveform) // chunk_size)
    features = []
    for i in range(n_chunks):
        chunk = waveform[i*chunk_size:(i+1)*chunk_size]
        if len(chunk) > 100:
            features.append([np.mean(chunk), np.std(chunk), np.max(chunk), np.min(chunk), np.sum(chunk**2)/len(chunk)])
    return np.array(features) if features else np.zeros((1,5))

def transcribe(path, lang="en"):
    return whisper_model.transcribe(str(path), language=lang)["text"]

print("✅ Functions ready")

---
# Part A: EATD Processing
*(User uploads data to Drive beforehand)*

In [ ]:
EATD_RAW = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/EATD-Corpus")
EATD_OUTPUT = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/EATD-Corpus")

if EATD_RAW.exists():
    raw = [d.name for d in EATD_RAW.iterdir() if d.is_dir() and d.name.startswith('t_')]
    existing = set([f.stem for f in EATD_OUTPUT.glob("*.h5")])
    remaining = [p for p in raw if p not in existing]
    print(f"📂 EATD: {len(raw)} total | {len(existing)} done | {len(remaining)} remaining")
else:
    print(f"⚠️ EATD not found at {EATD_RAW}")
    remaining = []

In [ ]:
eatd_ok = 0
for pid in tqdm(remaining, desc="EATD"):
    h5_path = EATD_OUTPUT / f"{pid}.h5"
    if h5_path.exists(): continue
    try:
        pdir = EATD_RAW / pid
        wavs = list(pdir.glob("*.wav"))
        if not wavs: continue
        
        all_wave, all_text = [], []
        for w in wavs:
            wave, _ = resample_audio(w)
            all_wave.append(wave)
            all_text.append(transcribe(w, "zh"))
        
        combined = np.concatenate(all_wave)
        label = 0
        lf = pdir / "label.txt"
        if lf.exists():
            try:
                sds = int(open(lf).read().strip().split()[0])
                label = 1 if sds >= 53 else 0
            except: pass
        
        with h5py.File(h5_path, 'w') as f:
            f.create_dataset('audio_features', data=extract_features(combined))
            f.create_dataset('transcript', data=" ".join(all_text).encode('utf-8'))
            f.create_dataset('label', data=label)
            f.attrs['source'] = 'EATD-Corpus'
        eatd_ok += 1
    except Exception as e:
        print(f"❌ {pid}: {e}")

print(f"\n✅ EATD processed: {eatd_ok}")

---
# Part B: MODMA Processing
**52 participants: 23 MDD + 29 HC**

In [ ]:
MODMA_RAW = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/MODMA_Raw")
MODMA_OUTPUT = Path("/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/MODMA")

print("="*60)
print("🔗 MODMA DATASET DOWNLOAD (2.5 GB)")
print("="*60)
print("\nDataset: 23 MDD + 29 HC = 52 participants")
print("Contents: Demographics + Psychological assessments")
print("\nEnter download link (Google Drive or direct URL):")

modma_link = input("📥 MODMA link (or Enter to skip): ").strip()

if modma_link:
    print("\n⬇️ Downloading...")
    if 'drive.google.com' in modma_link:
        import gdown
        if '/folders/' in modma_link:
            gdown.download_folder(modma_link, output="/content/modma_dl", quiet=False)
        else:
            gdown.download(modma_link, "/content/modma.zip", quiet=False, fuzzy=True)
            !unzip -q -o /content/modma.zip -d /content/modma_dl
    else:
        !wget -O /content/modma.zip "{modma_link}"
        !unzip -q -o /content/modma.zip -d /content/modma_dl
    
    # Move to Drive
    import shutil
    for f in Path("/content/modma_dl").rglob("*"):
        if f.is_file():
            shutil.copy(str(f), str(MODMA_RAW / f.name))
    print(f"\n✅ Saved to Drive")
else:
    print("⏭️ Using existing data")

In [ ]:
# Analyze MODMA structure
print("\n📂 MODMA structure:")
file_types = {}
for f in MODMA_RAW.rglob("*"):
    if f.is_file():
        ext = f.suffix.lower()
        file_types[ext] = file_types.get(ext, 0) + 1

for ext, count in sorted(file_types.items()):
    print(f"  {ext}: {count}")

# Find participant folders or files
audio_files = list(MODMA_RAW.rglob("*.wav")) + list(MODMA_RAW.rglob("*.mp3"))
print(f"\n🎵 Audio files: {len(audio_files)}")

In [ ]:
# MODMA has MDD (23) and HC (29) - detect from folder/filename
modma_existing = set([f.stem for f in MODMA_OUTPUT.glob("*.h5")])
to_process = [f for f in audio_files if f"modma_{f.stem}" not in modma_existing]

print(f"📋 To process: {len(to_process)}")

modma_ok = 0
for audio in tqdm(to_process, desc="MODMA"):
    pid = f"modma_{audio.stem}"
    h5_path = MODMA_OUTPUT / f"{pid}.h5"
    if h5_path.exists(): continue
    
    try:
        wave, _ = resample_audio(audio)
        text = transcribe(audio, "en")
        features = extract_features(wave)
        
        # Label: MDD=1, HC=0 (detect from path)
        path_lower = str(audio).lower()
        label = 1 if any(x in path_lower for x in ['mdd', 'patient', 'depressed', 'positive']) else 0
        
        with h5py.File(h5_path, 'w') as f:
            f.create_dataset('audio_features', data=features)
            f.create_dataset('transcript', data=text.encode('utf-8'))
            f.create_dataset('label', data=label)
            f.attrs['source'] = 'MODMA'
        modma_ok += 1
    except Exception as e:
        print(f"❌ {audio.name}: {e}")

print(f"\n✅ MODMA processed: {modma_ok}")

## Final Summary

In [ ]:
labels_data = []

for h5 in EATD_OUTPUT.glob("*.h5"):
    with h5py.File(h5, 'r') as f:
        labels_data.append({'Participant_ID': h5.stem, 'PHQ8_Binary': int(f['label'][()]), 'Source': 'EATD'})

for h5 in MODMA_OUTPUT.glob("*.h5"):
    with h5py.File(h5, 'r') as f:
        labels_data.append({'Participant_ID': h5.stem, 'PHQ8_Binary': int(f['label'][()]), 'Source': 'MODMA'})

df = pd.DataFrame(labels_data)
df['PHQ8_Score'] = df['PHQ8_Binary'].apply(lambda x: 15 if x else 3)
df.to_csv("/content/drive/MyDrive/DAIC-WOZ_Datasets/eatd_modma_labels.csv", index=False)

print(f"{'='*50}")
print(f"🏆 PROCESSING COMPLETE")
print(f"{'='*50}")
print(f"EATD: {len([x for x in labels_data if x['Source']=='EATD'])}")
print(f"MODMA: {len([x for x in labels_data if x['Source']=='MODMA'])}")
print(f"Total: {len(labels_data)}")